In [37]:
import os
import re
from dotenv import load_dotenv
from itertools import groupby
from functools import partial
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_classic.indexes import SQLRecordManager, index
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

True

In [38]:
# Load all PDFs from data folder
all_docs = []
data_dir = "data"
files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pdf")])

for file in files:
    # Extract chapter number from filename (e.g., book-chapter-00.pdf -> 00)
    match = re.search(r"book-chapter-(\d+)\.pdf", file)
    if match:
        chapter_num = match.group(1)
        loader = PyPDFLoader(os.path.join(data_dir, file))
        docs = loader.load()
        
        # Add metadata to each document (page)
        for doc in docs:
            doc.metadata.update({
                "volume": 1,
                "chapter": int(chapter_num)
            })
        all_docs.extend(docs)

# Split into smaller chunks
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Average characters per chunk
    chunk_overlap=200,   # Overlap for context across chunks
    add_start_index=True # Add chunk index in metadata for traceability
)
chunks = child_splitter.split_documents(all_docs)

In [39]:
print(f"Split into {len(chunks)} chunks")
print("keys:")
print(vars(chunks[0]).keys())  # First chunk
print("metadata sample (first chunk):") 
print(chunks[0].metadata)
print("metadata sample (last chunk):")
print(chunks[-1].metadata)
print("page_content sample:")
print(chunks[0].page_content[:200])

Split into 259 chunks
keys:
dict_keys(['id', 'metadata', 'page_content', 'type'])
metadata sample (first chunk):
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329105746', 'source': 'data\\book-chapter-00.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'volume': 1, 'chapter': 0, 'start_index': 0}
metadata sample (last chunk):
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329110040', 'source': 'data\\book-chapter-03.pdf', 'total_pages': 24, 'page': 23, 'page_label': '24', 'volume': 1, 'chapter': 3, 'start_index': 0}
page_content sample:
Speech and Language Processing
An Introduction to Natural Language Processing,
Computational Linguistics, and Speech Recognition
with Language Models
Third Edition draft
Daniel Jurafsky
Stanford Unive


## Analyzing the chunks

In [40]:
def group_docs_by_metadata(documents, key="source"):
    # Sort is required for groupby to work correctly
    sorted_docs = sorted(documents, key=lambda x: x.metadata.get(key, "unknown"))
    
    grouped = {
        k: list(g) 
        for k, g in groupby(sorted_docs, key=lambda x: x.metadata.get(key, "unknown"))
    }
    return grouped

In [41]:
chunks_by_chapter = group_docs_by_metadata(chunks, "chapter")
for chapter in chunks_by_chapter:
    print(f"Chunks of the chapter {chapter}: {len(chunks_by_chapter[chapter])}")

Chunks of the chapter 0: 31
Chunks of the chapter 1: 2
Chunks of the chapter 2: 136
Chunks of the chapter 3: 90


## Indexing

In [42]:
api_key = os.getenv("API_KEY")
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

In [43]:
get_vectorstore = partial(Chroma, embedding_function=embedding)

In [44]:
collection_name = "langchain_docs_index"

In [45]:
namespace = f"chroma/{collection_name}"
record_manager = SQLRecordManager(
    namespace=namespace,
    db_url="sqlite:///:memory:",
)
record_manager.create_schema()

### Without parent documents

In [46]:
# vector_store = get_vectorstore(
#     collection_name=collection_name,
# )

In [47]:
# index(
#     docs_source=chunks[0:3],
#     record_manager=record_manager,
#     vector_store=vector_store,
#     source_id_key="source",
#     cleanup="incremental",
# )

In [48]:
# # Testing incremental cleanup, trying to add the same chunks
# index(
#     docs_source=chunks[0:3],
#     record_manager=record_manager,
#     vector_store=vector_store,
#     source_id_key="source",
#     cleanup="incremental",
# )

### Parent Retriever

In [49]:
vectorstore = get_vectorstore(collection_name=f"{collection_name}_with_parents")
docstore = InMemoryStore()

In [50]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,     # Average characters per chunk
    chunk_overlap=800,   # Overlap for context across chunks
    add_start_index=True # Add chunk index in metadata for traceability
)

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [53]:
retriever.add_documents(all_docs)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 3.934494794s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', '

In [66]:
sub_docs = retriever.invoke("fundamental")
sub_docs

[]

In [67]:
vectorstore.similarity_search(
    "fundamental"
)

[]